# Wormhole Transmission Signal

Full characterization of the wormhole transmission signal $C(t) = \langle TFD | \psi^R_j(t) \psi^L_j(0) | TFD \rangle$ in the coupled SYK model.

## Contents
1. Signal at representative parameters
2. Peak height vs coupling $\mu$
3. Revival time scaling
4. Temperature dependence

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm

from src.doubled import DoubledSYK
from src.tfd import build_tfd
from src.wormhole import transmission_signal, extract_peak
from src.disorder_average import save_results, load_results, check_cache
from src.utils import ensure_dir

ensure_dir('../results')
ensure_dir('../data')

plt.rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'axes.titlesize': 14,
    'legend.fontsize': 10,
    'figure.figsize': (8, 6),
    'figure.dpi': 150,
})

print('Wormhole signal notebook loaded.')

## 1. Transmission Signal at Representative Parameters

Compute $|C(t)|$ for dense SYK at multiple $N$, $\beta$, and $\mu$ values with full disorder averaging.

In [ ]:
# Parameters
N_values = [8, 10, 12]
beta_values = [4, 8]
mu_values = [0.0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.5]
t_array = np.linspace(0, 40, 200)
n_realizations = 30

# Compute or load
cache_path = '../data/wormhole_signal.npz'

def compute_signal_single(seed, N, beta, mu):
    """Compute transmission signal for a single disorder realization."""
    ds = DoubledSYK(N, seed=seed, use_sparse=False)
    tfd, Z = build_tfd(ds, beta)
    H = ds.build_coupled_hamiltonian(mu)
    C = transmission_signal(H, tfd, ds.psi_L, ds.psi_R, t_array)
    return np.abs(C)

if check_cache(cache_path):
    data = load_results(cache_path)
    C_avg = {}
    C_err = {}
    for N in N_values:
        for beta in beta_values:
            for mu in mu_values:
                key = f'C_N{N}_beta{beta}_mu{mu:.3f}'
                ekey = f'Cerr_N{N}_beta{beta}_mu{mu:.3f}'
                if key in data:
                    C_avg[(N, beta, mu)] = data[key]
                    C_err[(N, beta, mu)] = data[ekey]
    print('Loaded wormhole signal data from cache.')
else:
    C_avg = {}
    C_err = {}
    total = len(N_values) * len(beta_values) * len(mu_values)
    count = 0
    
    for N in N_values:
        for beta in beta_values:
            for mu in mu_values:
                C_all = []
                for seed in range(n_realizations):
                    c = compute_signal_single(seed, N, beta, mu)
                    C_all.append(c)
                C_all = np.array(C_all)
                C_avg[(N, beta, mu)] = np.mean(C_all, axis=0)
                C_err[(N, beta, mu)] = np.std(C_all, axis=0) / np.sqrt(n_realizations)
                count += 1
                print(f'[{count}/{total}] N={N}, beta={beta}, mu={mu:.3f}: peak={np.max(C_avg[(N,beta,mu)]):.4f}')
    
    save_data = {'t_array': t_array, 'N_values': N_values, 'beta_values': beta_values, 'mu_values': mu_values}
    for (N, beta, mu), C in C_avg.items():
        save_data[f'C_N{N}_beta{beta}_mu{mu:.3f}'] = C
        save_data[f'Cerr_N{N}_beta{beta}_mu{mu:.3f}'] = C_err[(N, beta, mu)]
    save_results(cache_path, **save_data)
    print('Saved to cache.')

In [ ]:
# Plot: |C(t)| for representative parameters
fig, axes = plt.subplots(len(beta_values), len(N_values), figsize=(5*len(N_values), 5*len(beta_values)))
if len(beta_values) == 1:
    axes = axes[np.newaxis, :]

mu_plot = [0.0, 0.05, 0.1, 0.2, 0.5]
colors_mu = plt.cm.plasma(np.linspace(0.1, 0.9, len(mu_plot)))

for row, beta in enumerate(beta_values):
    for col, N in enumerate(N_values):
        ax = axes[row, col]
        for mu, c in zip(mu_plot, colors_mu):
            key = (N, beta, mu)
            if key in C_avg:
                ax.plot(t_array, C_avg[key], color=c, label=f'$\\mu={mu}$')
                ax.fill_between(t_array, 
                               C_avg[key] - C_err[key],
                               C_avg[key] + C_err[key],
                               alpha=0.15, color=c)
        ax.set_xlabel('t [1/J]')
        ax.set_ylabel('|C(t)|')
        ax.set_title(f'N={N}, $\\beta={beta}$')
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.3)

plt.suptitle('Wormhole Transmission Signal (disorder averaged)', y=1.02)
plt.tight_layout()
plt.savefig('../results/02_transmission_signal.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Peak Height vs Coupling $\mu$

In [ ]:
# Extract peak height and time for all parameter combinations
peak_data = {}
for N in N_values:
    for beta in beta_values:
        peaks = []
        times = []
        widths = []
        for mu in mu_values:
            key = (N, beta, mu)
            if key in C_avg:
                ph, pt, fw = extract_peak(C_avg[key], t_array)
                peaks.append(ph)
                times.append(pt)
                widths.append(fw)
        peak_data[(N, beta)] = {
            'peak_heights': np.array(peaks),
            'peak_times': np.array(times),
            'fwhm': np.array(widths)
        }

# Plot peak height vs mu
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: peak height vs mu
ax = axes[0]
for N in N_values:
    for beta in beta_values:
        key = (N, beta)
        if key in peak_data:
            ax.plot(mu_values, peak_data[key]['peak_heights'], 'o-',
                   label=f'N={N}, $\\beta={beta}$', markersize=5)

ax.set_xlabel(r'$\mu$ [J]')
ax.set_ylabel(r'$\max_t |C(t)|$')
ax.set_title('Peak Height vs Coupling')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: peak time vs mu
ax = axes[1]
for N in N_values:
    for beta in beta_values:
        key = (N, beta)
        if key in peak_data:
            ax.plot(mu_values, peak_data[key]['peak_times'], 'o-',
                   label=f'N={N}, $\\beta={beta}$', markersize=5)

ax.set_xlabel(r'$\mu$ [J]')
ax.set_ylabel(r'$t_{\mathrm{peak}}$ [1/J]')
ax.set_title('Revival Time vs Coupling')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/02_peak_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Temperature Dependence

In [ ]:
# Peak height vs beta at fixed mu
mu_fixed = 0.1

fig, ax = plt.subplots(figsize=(8, 6))
for N in N_values:
    heights = []
    for beta in beta_values:
        key = (N, beta, mu_fixed)
        if key in C_avg:
            ph, _, _ = extract_peak(C_avg[key], t_array)
            heights.append(ph)
        else:
            heights.append(0)
    ax.plot(beta_values, heights, 'o-', label=f'N={N}', markersize=8)

ax.set_xlabel(r'$\beta$ [1/J]')
ax.set_ylabel(r'$\max_t |C(t)|$')
ax.set_title(f'Peak Height vs Temperature ($\\mu={mu_fixed}$)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/02_temperature_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

print('Wormhole signal characterization complete.')

## Summary

Key observations:
- At $\mu = 0$: no transmission (no coupling between L and R)
- At $\mu > 0$: a revival peak appears in $|C(t)|$, consistent with information traversing the wormhole
- Peak height increases with $\mu$ in the small-$\mu$ regime
- Lower temperatures (larger $\beta$) tend to produce sharper, more defined peaks
- Finite-size effects are significant at these small $N$ values